# 01 · Data discovery & preprocessing
Dataset: `stanleyjzheng/rsna-str-tfrec` (RSNA-STR Pulmonary Embolism Detection, TFRecord format).
Add it to your Kaggle notebook via **Add Data**, or point `cfg.tfrec_glob` at your copy.

**What this notebook does**
1. Prints the TFRecord schema so you can confirm the field names (I could not inspect the files while writing this — **check cell 3 before running the rest**).
2. Decodes slices → groups by study → sorts by z → HU/intensity normalisation → resamples to `cfg.vol_shape`.
3. Runs noise/artifact QC, writes one `.npz` per study, and creates train/val/test splits (stratified, study-level, no leakage).

**Labels used** (RSNA-STR columns): `negative_exam_for_pe`, `rv_lv_ratio_gte_1`, `leftsided_pe`, `rightsided_pe`, `central_pe`, `chronic_pe`, and image-level `pe_present_on_image`.

In [ ]:
%run ./00_common.ipynb

In [ ]:
import tensorflow as tf
tf.config.set_visible_devices([], "GPU")          # keep GPU free for PyTorch
files = sorted(glob.glob(cfg.tfrec_glob, recursive=True))
print(len(files), "tfrec files"); print(files[:3])
raw = next(iter(tf.data.TFRecordDataset(files[:1]).take(1))).numpy()
ex = tf.train.Example.FromString(raw)
for k, v in ex.features.feature.items():
    kind = v.WhichOneof("kind"); vals = getattr(v, kind).value
    print(f"{k:32s} {kind:11s} n={len(vals):<7d}", (vals[0] if kind != "bytes_list" else f"<{len(vals[0])} bytes>") if len(vals) else "")

### Field map — edit this to match the printout above
`None` means "not present in the TFRecord". If you also have the official `train.csv`, set `cfg.labels_csv` and labels are read from it instead.

In [ ]:
FIELD = dict(
    study="StudyInstanceUID",           # study id (string/bytes)
    sop="SOPInstanceUID",               # image id (optional)
    image="image",                      # encoded image bytes
    z=None,                             # slice position / instance number if available (e.g. "z_pos"); else read order is used
    slice_label="pe_present_on_image",  # image-level PE flag
    study_labels=dict(negative_exam_for_pe="negative_exam_for_pe", rv_lv_ratio_gte_1="rv_lv_ratio_gte_1",
                      leftsided_pe="leftsided_pe", rightsided_pe="rightsided_pe", central_pe="central_pe", chronic_pe="chronic_pe"),
)
IMAGE_ENCODING = "jpeg"      # "jpeg" | "png" | "raw_uint8" | "raw_int16" (raw_int16 = Hounsfield units)
HU_WINDOW = (-100, 500)      # only used when IMAGE_ENCODING == "raw_int16"; CTPA-friendly window

import pandas as pd
CSV = None
if cfg.labels_csv and os.path.exists(cfg.labels_csv):
    CSV = pd.read_csv(cfg.labels_csv); print("labels csv:", CSV.shape)

def _get(ex, key):
    if key is None or key not in ex.features.feature: return None
    f = ex.features.feature[key]; return list(getattr(f, f.WhichOneof("kind")).value)

def _scalar(ex, key, default=np.nan):
    v = _get(ex, key); 
    if not v: return default
    v = v[0]; return v.decode() if isinstance(v, bytes) else v

def decode_image(buf, H, W):
    if IMAGE_ENCODING in ("jpeg", "png"):
        im = tf.io.decode_image(buf, channels=1, expand_animations=False)
        im = tf.image.resize(im, (H, W), method="area")[..., 0].numpy()
        return (im / 255.0).astype(np.float32)                                  # already windowed 8-bit
    if IMAGE_ENCODING == "raw_uint8":
        side = int(math.sqrt(len(buf))); im = np.frombuffer(buf, np.uint8).reshape(side, side)
    else:
        side = int(math.sqrt(len(buf) // 2)); hu = np.frombuffer(buf, np.int16).reshape(side, side).astype(np.float32)
        im = np.clip((hu - HU_WINDOW[0]) / (HU_WINDOW[1] - HU_WINDOW[0]), 0, 1)
    im = tf.image.resize(im[..., None].astype(np.float32), (H, W), method="area")[..., 0].numpy()
    return (im / 255.0 if IMAGE_ENCODING == "raw_uint8" else im).astype(np.float32)

### Volume construction & QC
* z-sorting (when `FIELD["z"]` exists), area-resize in-plane, linear resample in z.
* Slice labels are resampled with **max** pooling so that a PE slice is never lost.
* QC: noise (robust MAD of neighbouring-voxel differences), contrast, slice count, NaNs.

In [ ]:
from scipy import ndimage as ndi
D, H, W = cfg.vol_shape

def resample_z(vol, D):
    n = vol.shape[0]
    return vol if n == D else ndi.zoom(vol, (D / n, 1, 1), order=1).astype(np.float32)

def resample_labels(lab, D):
    n = len(lab); e = np.linspace(0, n, D + 1).astype(int)
    return np.array([lab[e[i]:max(e[i + 1], e[i] + 1)].max() for i in range(D)], np.float32)

def qc_metrics(vol, n_src):
    d = (vol[:, :, 1:] - vol[:, :, :-1]).ravel()[::7]
    noise = float(np.median(np.abs(d - np.median(d))) / 0.6745 / np.sqrt(2))
    body = vol[vol > 0.2]
    return dict(n_src_slices=int(n_src), noise=noise, contrast=float(np.percentile(body, 95) - np.percentile(body, 5)) if body.size else 0.,
                nan=bool(np.isnan(vol).any()), flag=bool(n_src < 50 or np.isnan(vol).any() or noise > 0.08 or body.size < 1000))

def study_labels(ex, sid):
    y = np.full(len(LABELS), np.nan, np.float32)
    if CSV is not None:
        r = CSV[CSV.StudyInstanceUID == sid]
        if len(r): r = r.iloc[0]; src = {k: float(r[k]) for k in FIELD["study_labels"] if k in r}
        else: src = {}
    else:
        src = {k: _scalar(ex, v) for k, v in FIELD["study_labels"].items()}
    g = lambda k: src.get(k, np.nan)
    y[0] = 1 - g("negative_exam_for_pe") if not np.isnan(g("negative_exam_for_pe")) else np.nan
    y[1:] = [g("rv_lv_ratio_gte_1"), g("leftsided_pe"), g("rightsided_pe"), g("central_pe"), g("chronic_pe")]
    return y

def finalize(sid, items, qc_rows):
    items.sort(key=lambda t: t[0])                                    # by z / read order
    sl = np.stack([t[1] for t in items]); lab = np.array([t[2] for t in items], np.float32)
    vol = resample_z(sl, D)
    q = qc_metrics(vol, len(items)); q["study"] = sid; qc_rows.append(q)
    y = items[0][3]
    if np.isnan(lab).all():
        sl_lab = np.full(D, np.nan, np.float32)
    else:
        sl_lab = resample_labels(np.nan_to_num(lab), D)
        if y[0] == 0: sl_lab[:] = 0                                   # negative exam => no PE slices
    np.savez_compressed(f"{cfg.vol_dir}/{sid}.npz", vol=vol.astype(np.float16), y=y, slice_lab=sl_lab)

def build_all(files, max_studies=None, max_open=4):
    open_, done, qc_rows, dup = {}, set(), [], 0
    ds = tf.data.TFRecordDataset(files, num_parallel_reads=2)
    for n_rec, raw in enumerate(ds):
        ex = tf.train.Example.FromString(raw.numpy())
        sid = _scalar(ex, FIELD["study"], "unknown")
        if CSV is None and sid in done: dup += 1; continue
        img = decode_image(_get(ex, FIELD["image"])[0], H, W)
        z = _scalar(ex, FIELD["z"], len(open_.get(sid, [])))
        pe = _scalar(ex, FIELD["slice_label"], np.nan)
        if CSV is not None and FIELD["sop"] and "pe_present_on_image" in CSV:
            sop = _scalar(ex, FIELD["sop"], None); r = CSV[CSV.SOPInstanceUID == sop]; pe = float(r.pe_present_on_image.iloc[0]) if len(r) else np.nan
        if sid not in open_: open_[sid] = []
        open_[sid].append((float(z), img, float(pe) if pe == pe else np.nan, study_labels(ex, sid) if not open_[sid] else open_[sid][0][3]))
        if len(open_) > max_open:                                       # records are normally contiguous per study
            old = next(iter(open_)); finalize(old, open_.pop(old), qc_rows); done.add(old)
            if max_studies and len(done) >= max_studies: break
        if n_rec % 5000 == 0: print(n_rec, "records |", len(done), "studies")
    for sid, it in open_.items():
        if not max_studies or len(done) < max_studies: finalize(sid, it, qc_rows); done.add(sid)
    print(f"done: {len(done)} studies; {dup} records skipped because their study had already been closed (non-contiguous records)")
    return pd.DataFrame(qc_rows)

qc = build_all(files, max_studies=None)        # set e.g. max_studies=200 for a quick smoke test
qc.to_csv(f"{cfg.out_dir}/qc.csv", index=False); print(qc.describe(), "\nflagged:", int(qc.flag.sum()))

### Splits (study-level, stratified on PE) and class balance

In [ ]:
from sklearn.model_selection import train_test_split
ids, ypos = [], []
for f in sorted(glob.glob(f"{cfg.vol_dir}/*.npz")):
    a = np.load(f); ids.append(os.path.basename(f)[:-4]); ypos.append(a["y"][0])
ids, ypos = np.array(ids), np.array(ypos)
good = ~np.isnan(ypos); print(f"{len(ids)} studies, {int((~good).sum())} without study label (excluded from splits)")
ids, ypos = ids[good], ypos[good]
flagged = set(qc.loc[qc.flag, "study"]); ymap = dict(zip(ids, ypos))
tr, te = train_test_split(ids, test_size=0.15, stratify=ypos, random_state=cfg.seed)
tr, va = train_test_split(tr, test_size=0.1765, stratify=[ymap[i] for i in tr], random_state=cfg.seed)
json.dump(dict(train=list(tr), val=list(va), test=list(te), qc_flagged=list(flagged)), open(f"{cfg.work}/splits.json", "w"))
print(len(tr), len(va), len(te), "| PE prevalence:", ypos.mean().round(3))

### Visual sanity check

In [ ]:
import matplotlib.pyplot as plt
sp = load_splits(); sid = sp["train"][0]; a = np.load(f"{cfg.vol_dir}/{sid}.npz")
v = a["vol"].astype(np.float32); fig, ax = plt.subplots(1, 4, figsize=(14, 3.5))
for i, z in enumerate(np.linspace(8, D - 8, 4).astype(int)):
    ax[i].imshow(v[z], cmap="gray", vmin=0, vmax=1); ax[i].set_title(f"z={z}  PE-slice={a['slice_lab'][z]}"); ax[i].axis("off")
plt.suptitle(f"{sid}  labels={dict(zip(LABELS, a['y']))}"); plt.show()